# Praktik Pengolahan Citra Digital
**Segmentasi Simbol Sekop ♠, Hati ♥, Wajik ♦, dan Keriting ♣ pada Kartu Remi**

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

# Helper function to show images in notebook
def show_images(images, titles, cmap='gray', cols=2):
    n = len(images)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(10 * cols, 5 * rows))
    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        if cmap == 'gray' and len(images[i].shape) == 2:
            plt.imshow(images[i], cmap='gray')
        else:
            plt.imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
        plt.title(titles[i])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

### Langkah 1: Upload dan Konversi Grayscale
Memuat gambar, mengubah ke grayscale, serta mengatur Brightness dan Contrast.

In [ ]:
# 1. Load Image
image_path = 'data/gambar.jpg'  # Sesuaikan dengan path gambar Anda
img_bgr = cv2.imread(image_path)

if img_bgr is None:
    print(f"Error: Gambar tidak ditemukan di {image_path}")
else:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    
    # Pengaturan Grayscale
    brightness = 0
    contrast = 1.0
    
    img_gray_adjusted = cv2.convertScaleAbs(img_gray, alpha=contrast, beta=brightness)
    
    show_images([img_rgb, img_gray, img_gray_adjusted], 
                ['Citra Asli (RGB)', 'Grayscale Default', f'Grayscale Adjusted\n(B={brightness}, C={contrast})'], cols=3)

### Langkah 2: Operasi Geometri Citra
Hanya menggunakan Cropping dan Flipping berdasarkan instruksi yang diperbarui.

In [ ]:
# Input dari Langkah 1
img_gray_input = img_gray_adjusted.copy()
img_rgb_input = img_rgb.copy()

h, w = img_gray_input.shape[:2]

# Pengaturan Geometri
# 1. Cropping
start_y, end_y = 0, h
start_x, end_x = 0, w

if start_y < end_y and start_x < end_x:
    img_gray_input = img_gray_input[start_y:end_y, start_x:end_x]
    img_rgb_input = img_rgb_input[start_y:end_y, start_x:end_x]

# 2. Flipping (0: Vertikal, 1: Horizontal, -1: Keduanya, None: Tidak ada)
flip_mode = None 

if flip_mode is not None:
    img_gray_input = cv2.flip(img_gray_input, flip_mode)
    img_rgb_input = cv2.flip(img_rgb_input, flip_mode)

# Simpan output ke variabel output langkah 2
img_gray_geo = img_gray_input
img_rgb_geo = img_rgb_input

show_images([img_rgb_geo, img_gray_geo], ['RGB Geometri', 'Grayscale Geometri'])

### Langkah 3: Deteksi Tepi Dinamis
Pilih salah satu metode deteksi tepi (Sobel ATAU Prewitt). Hasil garis tepi ini akan **diteruskan menjadi input Tahap 4**.

In [ ]:
# Input dari Langkah 2
gray_edge_input = img_gray_geo

# Pengaturan Deteksi Tepi
metode_pilihan = "Sobel" # Pilih antara "Sobel" atau "Prewitt"

# Parameter tetap (hardcoded)
ksize = 3
edge_thresh = 30

if metode_pilihan == "Sobel":
    sobel_x = cv2.Sobel(gray_edge_input, cv2.CV_64F, 1, 0, ksize=ksize)
    sobel_y = cv2.Sobel(gray_edge_input, cv2.CV_64F, 0, 1, ksize=ksize)
    edge_result = np.uint8(cv2.magnitude(sobel_x, sobel_y))
else:
    kernelx = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])
    kernely = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
    prewitt_x = cv2.filter2D(gray_edge_input, cv2.CV_64F, kernelx)
    prewitt_y = cv2.filter2D(gray_edge_input, cv2.CV_64F, kernely)
    edge_result = np.uint8(cv2.magnitude(prewitt_x, prewitt_y))

# Terapkan threshold filter
if edge_thresh > 0:
    edge_result = np.where(edge_result >= edge_thresh, edge_result, 0).astype(np.uint8)

# Simpan output garis tepi
img_edge = edge_result

show_images([gray_edge_input, img_edge], ['Input Grayscale', f'Deteksi Tepi: {metode_pilihan}'])


### Langkah 4: Segmentasi dari Garis Tepi
Mengambil gambar **garis tepi** dari Langkah 3, lalu menambal (fill) konturnya menjadi area solid, dan mengekstrak warna simbol dari RGB asli.

In [ ]:
# Input: Garis Tepi (Langkah 3) dan RGB (Langkah 2)
edge_seg_input = img_edge
rgb_seg_input = img_rgb_geo

# Parameter tetap (hardcoded)
min_area = 50
max_area = 15000

# 1. Threshold biner otomatis dengan Otsu
ret, thresh_edge = cv2.threshold(edge_seg_input, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Morphological closing untuk menyambung garis-garis tepi yang terputus
kernel = np.ones((3,3), np.uint8)
thresh_edge_closed = cv2.morphologyEx(thresh_edge, cv2.MORPH_CLOSE, kernel)

# 2. Cari kontur (garis luar) pada garis tepi biner yang sudah disambung
contours, hierarchy = cv2.findContours(thresh_edge_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 3. Tambal garis tepi menjadi area solid (fill) dan filter ukurannya
img_symbols_mask = np.zeros_like(edge_seg_input)
symbol_count = 0
if hierarchy is not None:
    for contour in contours:
        area = cv2.contourArea(contour)
        if min_area < area < max_area:
            # Menggambar kontur dengan ketebalan -1 akan menambal/mengisi area di dalamnya
            cv2.drawContours(img_symbols_mask, [contour], -1, 255, -1)
            symbol_count += 1

# 4. Terapkan mask padat yang sudah ditambal ke citra RGB asli
img_segmented_symbols = cv2.bitwise_and(rgb_seg_input, rgb_seg_input, mask=img_symbols_mask)

print(f'Threshold biner tepi otomatis: {ret:.0f}')
print(f'Simbol berhasil diekstrak: {symbol_count}')

show_images(
    [edge_seg_input, img_symbols_mask, img_segmented_symbols],
    ['Garis Tepi (Input L3)', 'Mask Area Solid (Hasil Fill)', f'Segmentasi Simbol ({symbol_count} area)'],
    cols=3
)
